# **Epoch-wise Deep Double Descent Demonstration**

[TBD add description, How it will be carried out, what to look out for]

In [ ]:
# Import Essential Modules
import wfdb
import ast
import pickle
import datetime
import numpy as np
import pandas as pd
import tensorflow as tf
from keras import layers
import matplotlib.pyplot as plt
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split

# config trail runs
PTBXL_PATH = "/home/student/Prathamesh's Project Pre-requisites/DataRes/physionet.org/files/ptb-xl/1.0.3/" # path to PTB-XL dataset
SAMPLING_RATE = 100 
NUM_CLASSES = 5
NOISE_RATE = 0.15 # 15% label noise (default) key for double descent
NUM_EPOCHS = 400
BATCH_SIZE = 64
SUBSET_SIZE = 3000 # small to easier interpolation
WIDTH = 64 # default width for epoch wise double descent

2026-02-25 16:05:16.099010: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-25 16:05:16.118981: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-25 16:05:16.798743: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


## Import Signal and Multi-label Data

In [ ]:
# Load data
def load_ptbxl(path, sampling_rate = 100):
    pass

In [ ]:
# Add label noise

def add_label_noise(y, noise_rate, num_classes, seed = 42):
    pass

## Over-parameterised Model Creation

[TBD add why it is created, maths behind it, how it is done, why chose the target of 900000 parameters]

In [ ]:
# Conv1D+BiLSTM Model

# residual block
def residual_block(x, filters, kernel_size = 3, stride=1):
    shortcut = x
    
    # main path
    x = layers.Conv1D(filters, kernel_size, strides = stride, padding = "same", use_bias = False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv1D(filters, kernel_size, strides = stride, padding = "same", use_bias = False)(x)
    x = layers.BatchNormalization()(x)

    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv1D(filters, kernel_size = 1, strides = stride, use_bias = False)(shortcut)
    shortcut = layers.BatchNormalization()(shortcut)

    # skip-connection
    x = layers.Add()([shortcut, x])
    x = layers.ReLU()(x)
    return x

def build_model(input_shape = (1000, 12), num_classes = 5, width = 64):
    # Input layers
    InputLayer = layers.Input(shape=input_shape)

    # 1D-CNN Block 1
    x = layers.Conv1D(width, kernel_size = 7, strides = 2, padding = "same", use_bias = False)(InputLayer)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    # residual block 1
    x = residual_block(x, width)
    # residual block 2
    x = residual_block(x, width)
    # residual block 3
    x = residual_block(x, width*2, stride = 2)
    # residual block 4
    x = residual_block(x, width*2)
    # residual block 5
    x = residual_block(x, width*4, stride = 2)
    # residual block 6
    x = residual_block(x, width*4)

    # Bi-LSTM Block 1
    x = layers.Bidirectional(layers.LSTM(units = 128, return_sequences = True))(x)
    x = layers.Bidirectional(layers.LSTM(units = 128, return_sequences = False))(x)

    # Classification Overhead
    x = layers.GlobalAveragePooling1D()(x)
    OutputLayer = layers.Dense(num_classes, activation = "sigmoid")(x)

    # Model Creation
    return tf.keras.Model(inputs = InputLayer, outputs = OutputLayer, name = "1DCNN_BiLSTM_Z_Model")

In [9]:
# Define custom metrics

# subclass custom Hamming Loss metric (Not using tensorflow addons here; version clash)
@tf.keras.utils.register_keras_serializable()
class HammingLoss(tf.keras.metrics.Metric):

    def __init__(self, name = "Hamming_loss", **kwargs):
        super(HammingLoss, self).__init__(name = name, **kwargs)
        self.total_mismatches = self.add_weight(name = "Total_mismatches", initializer = 'zeros', dtype = tf.float32)
        self.total_labels = self.add_weight(name = "Total_labels", initializer = 'zeros', dtype = tf.float32)

    def update_state(self, y_true, y_pred, sample_weight=None):
        # caste predictions and targets in tf.float32
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(tf.greater(y_pred, 0.5), tf.float32)

        # Calculate mismatches
        mismatches = tf.cast((tf.math.count_nonzero(tf.math.not_equal(y_true, y_pred), axis=-1)), tf.float32)

        # Find number of labels and batch size
        num_label = tf.cast(tf.shape(y_true)[-1], tf.float32) # shape is (rows, columns) and columns = number of elements in array
        batch_size = tf.cast(tf.shape(y_true)[0], tf.float32) # shape is (rows, columns) and rows = batch size per array
        
        # Update number of mismatches and total labels count
        self.total_mismatches.assign_add(tf.reduce_sum(mismatches)) # reduce sum adds all the elements in an array (here, instance)
        self.total_labels.assign_add(batch_size * num_label) # total label count = number of labels x batch size per instance

    def result(self):
        return self.total_mismatches / self.total_labels # Hamming Loss formula
    
    def reset_state(self): # reset atttributes
        self.total_mismatches.assign(0.)
        self.total_labels.assign(0.)

In [ ]:
# Custom callback save metrics in a list
class EpochHistory(tf.keras.callbacks.Callback):
    def __init__(self, val_data):
        super().__init__()
        self.val_data = val_data # (X_val, y_val)
        self.train_losses = []
        self.val_losses = []
        self.val_accs = []
        self.val_hl = []
    
    def on_epoch_end(self, epoch, logs = None):
        self.train_losses.append(logs["loss"])
        self.val_losses.append(logs["val_loss"])
        self.val_accs.append(logs["val_accuracy"]) # compile with binary accuracy
        self.val_hl.append(logs["val_Hamming_loss"])

        if (epoch + 1) % 50 == 0:
            print(f"Epoch {epoch+1} | train_loss={logs['loss']:.4f} "
                f"val_Hamming_loss={logs["val_Hamming_loss"]:.4f} val_loss={logs['val_loss']:.4f} val_acc={logs['val_accuracy']:.4f}")


## Training Run

In [ ]:
def run_experiment(noise_rate = 0.15, width = 64, subset_size = 3000, epochs = 400):

    # Load ptbxl

    # subset
    rng = np.random.default_rng(0)
    idx = rng.choice(len(x), size = min(subset_size, len(x)), replace = False)
    X, Y = X[idx], Y[idx]

    # Normalise per-lead
    X = (X - X.mean(axis = 1, keepdims = True)) / (X.std(axis = 1, keepdims = True) + 1e-8)

    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, stratify=Y, random_state=42
    )

    # Add label noise to training set only
    y_train_noisy = add_label_noise(y_train, noise_rate, NUM_CLASSES)
    print(f"Noise rate: {noise_rate} | Noisy labels: {(y_train_noisy != y_train).sum()}/{len(y_train)}")

    # Build model
    model = build_model(
    input_shape=(X_train.shape[1], X_train.shape[2]),
    num_classes=NUM_CLASSES,
    width=width
    )
    model.summary()

    # No weight decay, SGD with momentum (helps double descent appear)
    optimizer = tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.9)

    model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.binary_crossentropy(from_logits=True),
    metrics=[tf.keras.metrics.BinaryAccuracy(name = 'accuracy'), HammingLoss()]
    )

    # Cosine LR decay
    lr_scheduler = tf.keras.callbacks.LearningRateScheduler(
    lambda epoch: 0.01 * 0.5 * (1 + np.cos(np.pi * epoch / epochs))
    )

    history_cb = EpochHistory(val_data=(X_test, y_test))

    # Train
    model.fit(
    X_train, y_train_noisy,
    validation_data=(X_test, y_test),
    epochs=epochs,
    batch_size=BATCH_SIZE,
    callbacks=[history_cb, lr_scheduler],
    verbose=0
    )

    return history_cb

